In [12]:
!pip install -U transformers

In [ ]:
#Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model= "ibm-granite/granite-3.3-2b-instruct")

messages = [

{"role": "user", "content":"We are you?"},
]

pipe(messages)

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
#Load model directly

from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-3.3-2b-instruct")

model = AutoModelForCausalLM.from_pretrained("ibm-granite/granite-3.3-2b-instruct")

messages = [

{"role": "user", "content":"Who are you?"},
]

inputs = tokenizer.apply_chat_template (

messages,

add_generation_prompt=True,

tokenize=True,
return_dict=True,

return_tensors="pt",

).to(model.device)

outputs= model.generate(**inputs, max_new_tokens=40)

print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
import gradio as gr
from transformers import pipeline
import torch
import ast
import re
from datetime import datetime
from pathlib import Path
import json
!pip install black autopep8
!pip install reportlab
import black
import autopep8
from reportlab.lib.pagesizes import letter, A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
import requests
import os

class ChatbotConfig:
    MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"
    DEFAULT_TEMP = 0.7
    DEFAULT_MAX_TOKENS = 256
    DEFAULT_TOP_P = 0.95

conversation_history = []
model_pipe = None

def load_model():
    global model_pipe
    print("🚀 Loading IBM Granite 3.3 2B Instruct model...")
    device = 0 if torch.cuda.is_available() else -1
    device_type = "GPU" if device == 0 else "CPU"
    print(f"   Using: {device_type}")
    model_pipe = pipeline(
        "text-generation",
        model=ChatbotConfig.MODEL_NAME,
        device=device,
        torch_dtype=torch.float16 if device == 0 else torch.float32,
    )
    print("✅ Model loaded successfully!")
    return model_pipe

def detect_python_code(text):
    code_patterns = [r'def\s+\w+\s*\(', r'class\s+\w+', r'import\s+\w+', r'from\s+\w+\s+import', r'if\s+.*:', r'for\s+.*\s+in\s+', r'while\s+.*:']
    return any(re.search(pattern, text) for pattern in code_patterns)

def check_syntax(code):
    try:
        ast.parse(code)
        return True, None
    except SyntaxError as e:
        return False, f"Syntax Error (line {e.lineno}): {e.msg}"
    except Exception as e:
        return False, f"Error: {str(e)}"

def fix_code_indentation(code):
    lines = code.split('\n')
    fixed_lines = [line.rstrip().replace('\t', '    ') for line in lines]
    return '\n'.join(fixed_lines)

def fix_code_brackets(code):
    open_count = code.count('(') - code.count(')')
    if open_count > 0:
        code += ')' * open_count
    open_count = code.count('[') - code.count(']')
    if open_count > 0:
        code += ']' * open_count
    open_count = code.count('{') - code.count('}')
    if open_count > 0:
        code += '}' * open_count
    return code

def format_code(code):
    try:
        return black.format_str(code, mode=black.FileMode())
    except:
        try:
            return autopep8.fix_code(code)
        except:
            return code

def fix_python_code(code):
    code = fix_code_indentation(code)
    code = fix_code_brackets(code)
    is_valid, error_msg = check_syntax(code)
    if not is_valid:
        return {'success': False, 'fixed_code': code, 'error': error_msg, 'message': f"⚠️ Could not fully fix code: {error_msg}"}
    code = format_code(code)
    return {'success': True, 'fixed_code': code, 'error': None, 'message': "✅ Code fixed and formatted successfully!"}

def generate_response(user_message, temperature, max_tokens, top_p):
    conversation_history.append({"role": "user", "content": user_message})
    try:
        response = model_pipe(conversation_history, max_new_tokens=int(max_tokens), temperature=temperature, do_sample=True, top_p=top_p)
        assistant_message = response[0]['generated_text']
        if isinstance(assistant_message, list):
            assistant_message = assistant_message[-1].get("content", str(assistant_message))
        conversation_history.append({"role": "assistant", "content": assistant_message})
        return assistant_message
    except Exception as e:
        error_response = f"❌ Error: {str(e)}"
        conversation_history.append({"role": "assistant", "content": error_response})
        return error_response

def process_user_input(user_message, temperature, max_tokens, top_p, chat_history):
    if not user_message.strip():
        return chat_history
    if detect_python_code(user_message):
        code_pattern = r'```python\n(.*?)\n```|```\n(.*?)\n```'
        code_match = re.search(code_pattern, user_message, re.DOTALL)
        if code_match:
            code_snippet = code_match.group(1) or code_match.group(2)
            fix_result = fix_python_code(code_snippet)
            if fix_result['success']:
                response_text = f"{fix_result['message']}\n\n**Fixed Code:**\n```python\n{fix_result['fixed_code']}\n```"
            else:
                response_text = f"{fix_result['message']}\n\n**Original Code:**\n```python\n{code_snippet}\n```\n\n**Attempted Fix:**\n```python\n{fix_result['fixed_code']}\n```"
            chat_history.append((user_message, response_text))
            ai_response = generate_response(f"I fixed this Python code:\n```python\n{fix_result['fixed_code']}\n```\nBriefly explain what it does.", temperature, max_tokens, top_p)
            chat_history.append(("Analysis", ai_response))
            return chat_history
    ai_response = generate_response(user_message, temperature, max_tokens, top_p)
    chat_history.append((user_message, ai_response))
    return chat_history

def clear_chat():
    global conversation_history
    conversation_history = []
    return []

def download_txt():
    if not conversation_history:
        return "❌ Chat is empty!", None

    content = f"Granite Chatbot - Complete Chat History\nExported: {datetime.now()}\n{'='*80}\n\n"

    for i, msg in enumerate(conversation_history, 1):
        role = msg['role'].upper()
        content_text = msg['content']
        content += f"\n[Message {i}]\n{role}:\n{'-'*80}\n{content_text}\n{'-'*80}\n"

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"chat_history_{timestamp}.txt"

    try:
        Path(filename).write_text(content)
        return f"✅ Downloaded: {filename}", filename
    except Exception as e:
        return f"❌ Error: {str(e)}", None

def download_pdf():
    if not conversation_history:
        return "❌ Chat is empty!", None

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"chat_history_{timestamp}.pdf"

    try:
        doc = SimpleDocTemplate(filename, pagesize=A4, topMargin=0.5*inch, bottomMargin=0.5*inch)
        styles = getSampleStyleSheet()
        story = []

        # Title style
        title_style = ParagraphStyle(
            'CustomTitle',
            parent=styles['Heading1'],
            fontSize=20,
            textColor='#1e3c72',
            spaceAfter=12,
            alignment=1,
            fontName='Helvetica-Bold'
        )

        # User message style
        user_style = ParagraphStyle(
            'UserMessage',
            parent=styles['Normal'],
            fontSize=10,
            textColor='#0033CC',
            spaceAfter=8,
            spaceBefore=8,
            borderColor='#0033CC',
            borderWidth=1,
            borderPadding=8,
            leftIndent=20,
            rightIndent=20,
            backColor='#E6F0FF'
        )

        # Assistant message style
        assistant_style = ParagraphStyle(
            'AssistantMessage',
            parent=styles['Normal'],
            fontSize=10,
            textColor='#7b1fa2',
            spaceAfter=8,
            spaceBefore=8,
            borderColor='#7b1fa2',
            borderWidth=1,
            borderPadding=8,
            leftIndent=20,
            rightIndent=20,
            backColor='#F3E5F5'
        )

        # Header
        story.append(Paragraph("💬 Granite Chatbot - Complete Chat History", title_style))
        story.append(Paragraph(f"<b>Exported on:</b> {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", styles['Normal']))
        story.append(Paragraph(f"<b>Total Messages:</b> {len(conversation_history)}", styles['Normal']))
        story.append(Spacer(1, 0.3*inch))

        # Add each message
        for i, msg in enumerate(conversation_history, 1):
            role = msg['role'].upper()
            content_text = msg['content']

            # Clean HTML characters
            clean_text = content_text.replace('<', '&lt;').replace('>', '&gt;')

            # Add message number
            story.append(Paragraph(f"<b>Message #{i}</b> - <b style='color: #FF6B6B'>{role}</b>", styles['Normal']))

            # Add content with appropriate styling
            if msg['role'] == 'user':
                story.append(Paragraph(clean_text, user_style))
            else:
                story.append(Paragraph(clean_text, assistant_style))

            story.append(Spacer(1, 0.2*inch))

            # Add page break after every 3 messages to keep content readable
            if i % 3 == 0 and i != len(conversation_history):
                story.append(PageBreak())

        # Footer
        story.append(Spacer(1, 0.3*inch))
        story.append(Paragraph(f"<b>End of Chat History</b> | Generated by Granite Chatbot", styles['Normal']))

        doc.build(story)
        return f"✅ Downloaded: {filename}", filename
    except Exception as e:
        return f"❌ Error: {str(e)}", None

def generate_share_link():
    if not conversation_history:
        return "❌ Chat is empty!"
    try:
        chat_content = ""
        for msg in conversation_history:
            role = msg['role'].upper()
            content = msg['content']
            chat_content += f"{role}:\n{content}\n\n"
        pastebin_api = "https://pastebin.com/api/api_post.php"
        paste_data = {
            'api_dev_key': '',
            'api_option': 'paste',
            'api_paste_code': chat_content,
            'api_paste_name': f'Granite Chat {datetime.now().strftime("%Y-%m-%d %H:%M")}',
            'api_paste_expire_date': 'N',
            'api_paste_private': '0'
        }
        response = requests.post(pastebin_api, data=paste_data, timeout=5)
        if 'pastebin.com' in response.text:
            share_link = response.text.strip()
            return f"✅ SHAREABLE LINK!\n\n🔗 {share_link}\n\nShare on Twitter, Facebook, Instagram, WhatsApp, Telegram, Reddit!"
    except:
        pass
    summary = f"Messages: {len(conversation_history)}\nDate: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
    return f"✅ SHAREABLE SUMMARY:\n\n{summary}\n\nDownload TXT/PDF to share!"

load_model()

with gr.Blocks(title="Granite Chatbot", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 💬 Granite Chatbot\n### Chat + Python Code Fixer - No API Keys")
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label="Chat History", height=400, show_copy_button=True)
            with gr.Row():
                msg = gr.Textbox(label="Message", placeholder="Type message or paste Python code...", lines=2, scale=4)
                submit_btn = gr.Button("Send", scale=1, size="lg", variant="primary")
            with gr.Row():
                temperature = gr.Slider(label="🌡️ Temp", minimum=0.1, maximum=1.0, value=0.7, step=0.05)
                max_tokens = gr.Slider(label="📝 Tokens", minimum=50, maximum=1024, value=256, step=50)
                top_p = gr.Slider(label="🎯 Top-P", minimum=0.1, maximum=1.0, value=0.95, step=0.05)
            submit_btn.click(process_user_input, inputs=[msg, temperature, max_tokens, top_p, chatbot], outputs=[chatbot])
            msg.submit(process_user_input, inputs=[msg, temperature, max_tokens, top_p, chatbot], outputs=[chatbot])
        with gr.Column(scale=1):
            gr.Markdown("### 🎮 Controls")
            clear_btn = gr.Button("🗑️ Clear Chat", variant="stop", size="sm")
            clear_btn.click(clear_chat, outputs=[chatbot])
            gr.Markdown("---")
            gr.Markdown("### 📥 Download")

            txt_btn = gr.Button("📄 Download TXT", size="sm", variant="primary")
            pdf_btn = gr.Button("📋 Download PDF", size="sm", variant="primary")

            txt_file = gr.File(label="TXT File", visible=True)
            pdf_file = gr.File(label="PDF File", visible=True)

            status = gr.Textbox(label="Status", interactive=False, lines=2)

            txt_btn.click(download_txt, outputs=[status, txt_file])
            pdf_btn.click(download_pdf, outputs=[status, pdf_file])

            gr.Markdown("---")
            gr.Markdown("### 🔗 Share Link")
            share_btn = gr.Button("🚀 Generate Share Link", size="sm", variant="secondary")
            share_output = gr.Textbox(label="Share Link", interactive=False, lines=5)
            share_btn.click(generate_share_link, outputs=[share_output])

print("\n🚀 GRANITE CHATBOT LAUNCHED!\n")
demo.launch(share=True, show_error=True, show_api=False)